# Construcción y Estratificación del Dataset de Evaluación (sample_1000.csv)

Este notebook documenta paso a paso el pipeline de generación del dataset de prueba ciego acordado con el docente (**REQ-07**):
1. **Fuente de datos:** Lectura del archivo oficial de MITRE / CVEProject (`all_cves.zip`, ~630 MB, más de 250k CVEs).
2. **Extracción y Partición Temporal:** Filtrado estricto de CVEs publicados en el **segundo semestre de 2024** (`datePublished >= 2024-07-01`) para garantizar casos no vistos por los modelos ni contaminados en el preentrenamiento.
3. **Limpieza:** Descarte de registros sin descripción en inglés, sin severidad CVSS v3.x válida, o con textos menores a 15 caracteres.
4. **Filtros de Duplicados:** Eliminación de descripciones idénticas (`drop_duplicates`).
5. **Muestreo Estratificado:** Selección aleatoria determinista (`random_state=42`) respetando las proporciones poblacionales de severidad:
   - `MEDIUM`: 45.0% (450 casos)
   - `HIGH`: 37.0% (370 casos)
   - `CRITICAL`: 16.4% (164 casos)
   - `LOW`: 1.6% (16 casos)
6. **Exportación:** Generación del artefacto reproducible `sample_1000.csv`.

## 1. Configuración de Rutas y Dependencias

In [ ]:
import io
import json
import zipfile
import pandas as pd
import numpy as np

ZIP_PATH = "/media/vub/25f61be1-310a-49c4-bce7-a8bc793230e32/vub/postgrados/unlu/diplomado-ia/curso2-infra/propuesta/data/all_cves.zip"
OUTPUT_CSV = "sample_1000.csv"
RANDOM_SEED = 42

print("Dependencias listas.")

## 2. Extracción de CVEs de 2024-H2 desde el Zip Maestro
Filtramos únicamente registros de `2024` con fecha de publicación entre julio y diciembre de 2024.

In [ ]:
records = []

with zipfile.ZipFile(ZIP_PATH, "r") as outer:
    with outer.open("cves.zip") as nested:
        with zipfile.ZipFile(io.BytesIO(nested.read()), "r") as inner:
            for name in inner.namelist():
                if not (name.endswith(".json") and "/2024/" in name):
                    continue
                try:
                    with inner.open(name) as f:
                        data = json.load(f)
                    meta = data.get("cveMetadata", {})
                    date_pub = meta.get("datePublished", "")
                    # Particion temporal: segundo semestre 2024 (julio a diciembre)
                    if not any(date_pub.startswith(f"2024-{m:02d}") for m in range(7, 13)):
                        continue
                    cve_id = meta.get("cveId")
                    cna = data.get("containers", {}).get("cna", {})
                    # Descripcion en ingles
                    desc_text = ""
                    for d in cna.get("descriptions", []):
                        if d.get("lang") in ("en", "en-US", ""):
                            desc_text = d.get("value", "")
                            break
                    if not desc_text and cna.get("descriptions"):
                        desc_text = cna.get("descriptions")[0].get("value", "")
                    # Metricas CVSS 3.x
                    score_v3 = None
                    sev = None
                    for m in cna.get("metrics", []):
                        v3 = m.get("cvssV3_1") or m.get("cvssV3_0")
                        if v3:
                            score_v3 = v3.get("baseScore")
                            sev = (v3.get("baseSeverity") or "").upper()
                            break
                    # CWE
                    cwe_id = "UNKNOWN"
                    for pt in cna.get("problemTypes", []):
                        for desc in pt.get("descriptions", []):
                            if desc.get("cweId"):
                                cwe_id = desc.get("cweId")
                                break
                    if cve_id and desc_text and sev:
                        records.append({
                            "CVE-ID": cve_id,
                            "DATE_PUB": date_pub,
                            "CVSS-V4": None,
                            "CVSS-V3": score_v3,
                            "CVSS-V2": None,
                            "SEVERITY": sev,
                            "DESCRIPTION": desc_text,
                            "CWE-ID": cwe_id,
                        })
                except Exception:
                    continue

df_raw = pd.DataFrame(records)
print(f"Total CVEs extraidos de 2024-H2: {len(df_raw)}")
df_raw.head()

## 3. Limpieza y Filtros de Calidad
- Filtrar severidades fuera de las 4 clases válidas de CVSS (`LOW`, `MEDIUM`, `HIGH`, `CRITICAL`).
- Descartar descripciones vacías o truncadas (< 15 caracteres).
- Eliminar descripciones duplicadas para evitar fuga y sesgos repetitivos.

In [ ]:
valid_classes = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}
df_clean = df_raw[df_raw["SEVERITY"].isin(valid_classes)].copy()
df_clean = df_clean.dropna(subset=["DESCRIPTION", "SEVERITY"])
df_clean = df_clean[df_clean["DESCRIPTION"].str.strip().str.len() >= 15]
total_antes_dups = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["DESCRIPTION"])
print(f"Filas antes de deduplicar: {total_antes_dups}")
print(f"Filas despues de deduplicar: {len(df_clean)}")
print("\nDistribucion poblacional en el universo 2024-H2 limpio:")
print(df_clean["SEVERITY"].value_counts(normalize=True).round(4) * 100)

## 4. Muestreo Estratificado (Criterios exactos de la propuesta)
Extraemos una muestra de 1.000 observaciones fijando `random_state=42` para que sea 100% reproducible:
- `MEDIUM`: 450 casos (45.0%)
- `HIGH`: 370 casos (37.0%)
- `CRITICAL`: 164 casos (16.4%)
- `LOW`: 16 casos (1.6%)
Total: exactamente 1.000 casos.

In [ ]:
target_counts = {
    "MEDIUM": 450,
    "HIGH": 370,
    "CRITICAL": 164,
    "LOW": 16,
}

sample_parts = []
for sev, count in target_counts.items():
    subset = df_clean[df_clean["SEVERITY"] == sev]
    sampled_subset = subset.sample(n=count, random_state=RANDOM_SEED)
    sample_parts.append(sampled_subset)

df_sample = pd.concat(sample_parts)
# Barajar aleatoriamente el dataset final para no dejarlo ordenado por severidad
df_sample = df_sample.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

# Asignar ID secuencial de tracking institucional
df_sample["ID"] = range(260001, 260001 + len(df_sample))

# Reordenar columnas finales estandarizadas del curso
cols = ["ID", "CVE-ID", "CVSS-V4", "CVSS-V3", "CVSS-V2", "SEVERITY", "DESCRIPTION", "CWE-ID"]
df_sample = df_sample[cols]

print(f"Muestra estratificada final generada: {len(df_sample)} filas.")
print("\nConteo exacto por severidad:")
print(df_sample["SEVERITY"].value_counts())
print("\nPorcentajes:")
print((df_sample["SEVERITY"].value_counts(normalize=True) * 100).round(2))
df_sample.head(10)

## 5. Exportación del Dataset Final
Guardamos el archivo CSV para su uso en entrenamiento, evaluación y testing del pod.

In [ ]:
df_sample.to_csv(OUTPUT_CSV, index=False)
df_sample.to_csv("../model/sample_1000.csv", index=False)
print(f"Archivo guardado exitosamente en: {OUTPUT_CSV} y en ../model/sample_1000.csv")